# Telco Customer Churn Prediction — ICT6001 Week 01 Group Capstone

**Course:** ICT6001 — AI Programme (Week 1 Capstone)  
**Group:** Group 3 | **Members:** Joyce Yeo, KhoonSeng, Darren  
**Repo:** <To be updated>  
**Dataset:** Telco Customer Churn (594,194 rows × 21 features)  
**Target:** `Churn` (binary — 1 = churned, 0 = retained)

---

### Notebook Roadmap

| Section | Part | CRISP-DM Phase | Description |
|---------|------|----------------|-------------|
| §1 | A | Data Preparation | Advanced Data Prep & Pipeline Engineering |
| §2 | B | Modelling | Champion Model Selection via 5-fold CV |
| §3 | C | Modelling | Controlled Ablations & Tuning |
| §4 | E | Deployment | Business Decision Making |

## Set-up Environment

In [ ]:
%matplotlib inline
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, recall_score,
    balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="Set2")
print("Environment ready.")

In [ ]:
# Portable data load — works locally and in CI/headless runs.
# Set CAPSTONE_SAMPLE=N for fast iteration (~50k); 0 = full 594k.
DATA_DIR = os.environ.get("CAPSTONE_DATA", "data")
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

_n = int(os.environ.get("CAPSTONE_SAMPLE", "0"))
if _n:
    df = df.groupby("Churn", group_keys=False).sample(
        frac=min(1.0, _n / len(df)), random_state=RANDOM_STATE
    )
    print(f"[DEV MODE] stratified sample: {len(df):,} rows")
else:
    print(f"[FULL DATA] {len(df):,} rows")

df.shape

In [ ]:
# Churn → 0/1; coerce TotalCharges blanks to NaN (guard for test-set edge cases).
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

X = df.drop(columns=["id", "Churn"])
y = df["Churn"]

# 80 / 20 stratified split.  X_test is locked until §4 (touched exactly once).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"train : {X_train.shape}   test : {X_test.shape}")
print(f"Churn rate — train: {y_train.mean():.3f}   test: {y_test.mean():.3f}")

---
## 1. Advanced Data Preparation & Pipeline Engineering — Part A
**CRISP-DM Phase:** Data Preparation

**Purpose:** Finalise feature engineering and build a leakage-safe preprocessing pipeline.
All transforms (imputation, encoding, scaling, SMOTE oversampling) are encapsulated inside
a formal `sklearn.pipeline.Pipeline` so that no information from validation folds leaks
into the training distribution.


### 1.1 Exploratory Data Analysis

**Purpose:** Surface key patterns that inform feature decisions downstream.

In [ ]:
# Class distribution
churn_counts = y.value_counts().sort_index()
pct = (churn_counts / len(y) * 100).round(1)
print(churn_counts.rename({0: "No Churn", 1: "Churn"}).to_string())
print(f"\nChurn rate: {pct[1]:.1f}%  → moderate class imbalance; SMOTE addresses this in §1.3")

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(["No Churn (0)", "Churn (1)"], churn_counts.values,
              color=["#27ae60", "#e74c3c"], edgecolor="white")
for bar, val in zip(bars, churn_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f"{val:,}\n({val/len(y)*100:.1f}%)", ha="center", va="bottom", fontsize=9)
ax.set_title("Churn Distribution", fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Tenure distribution
for label, color in [(0, "#27ae60"), (1, "#e74c3c")]:
    subset = df[df["Churn"] == label]["tenure"]
    axes[0].hist(subset, bins=30, alpha=0.55, color=color,
                 label=f"Churn={label}", density=True)
axes[0].set_title("Tenure vs Churn", fontweight="bold")
axes[0].set_xlabel("Tenure (months)"); axes[0].legend()

# 2. MonthlyCharges boxplot
df_plot = df.copy(); df_plot["Churn"] = df_plot["Churn"].map({0: "No", 1: "Yes"})
df_plot.boxplot(column="MonthlyCharges", by="Churn", ax=axes[1],
                medianprops=dict(color="#e74c3c", linewidth=2))
axes[1].set_title("MonthlyCharges vs Churn", fontweight="bold")
axes[1].set_xlabel("Churn"); plt.sca(axes[1]); plt.title("")

# 3. Contract type
contract_churn = df.groupby(["Contract", "Churn"]).size().unstack(fill_value=0)
contract_churn.plot(kind="bar", ax=axes[2],
                    color=["#27ae60", "#e74c3c"], edgecolor="white", rot=30)
axes[2].set_title("Contract Type vs Churn", fontweight="bold")
axes[2].set_xlabel(""); axes[2].legend(["No Churn", "Churn"])

plt.suptitle("Key Drivers of Churn (EDA)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

print('''
EDA Insights
  1. Tenure: New customers (<12 months) have highest churn risk.
             Loyalty builds over time -- retention focus on the first year.
  2. MonthlyCharges: Churners pay ~$15-20 more per month (pricing sensitivity).
  3. Contract type: Month-to-month customers churn at ~42%;
                    two-year contract customers at <3%. Upgrading contracts is
                    the strongest single intervention lever.
''')

### 1.2 Missing Values & Data Quality

**Purpose:** Confirm completeness. `TotalCharges` was coerced to numeric above to guard against blank-string entries common in the raw telco dataset.

In [ ]:
missing = df.isnull().sum()
tc_nulls = missing["TotalCharges"]
other_nulls = missing.drop("TotalCharges").sum()
print(f"TotalCharges NaNs after coercion : {tc_nulls}")
print(f"All other columns missing        : {other_nulls}")
print("\nConclusion: no structural missing data.")

### 1.3 Feature Typing & Leakage-Safe Pipeline

**Purpose:** Encapsulate all transforms in a `ColumnTransformer → ImbPipeline`.
SMOTE is placed *inside* the pipeline — oversampling occurs only on training folds;
validation folds are never oversampled.

**Pipeline architecture:**
```
ImbPipeline
├── preprocessor (ColumnTransformer)
│   ├── num_pipe : StandardScaler
│   └── cat_pipe : OneHotEncoder(handle_unknown=ignore)
│                  → SelectPercentile(chi2, percentile=50)
├── smote       : SMOTE(random_state=42)   ← touches training fold only
└── classifier  : <model>
```

`SelectPercentile(chi2, 50)` retains the top-50% of one-hot columns by χ² association
with the target — reducing dimensionality and discarding low-signal binary dummies.

In [ ]:
num_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
cat_features = [c for c in X.columns if c not in num_features]

print(f"Numeric     ({len(num_features)}): {num_features}")
print(f"Categorical ({len(cat_features)}): {cat_features}")

num_pipe = Pipeline([
    ("scaler",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ("selector", SelectPercentile(score_func=chi2, percentile=50)),
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features),
], remainder="drop")

print("\nPreprocessor built. SMOTE will be added per-model in §2.")

---
## 2. Champion Model Selection — Part B
**CRISP-DM Phase:** Modelling

### 2.1 Primary Evaluation Metric

**Why ROC-AUC?**
- The dataset is imbalanced (~22.5% churn). Accuracy rewards the majority class.
- ROC-AUC measures a model's ability to **rank** churners above non-churners across
  *all* thresholds — threshold-independent, so it is a fair between-family comparison.
- It directly supports §4's threshold-shift analysis: higher AUC = more headroom
  to tune the threshold for the business's specific FN cost structure.
- **Recall** is tracked as a secondary metric: it captures how many real churners the
  model catches, directly reflecting the business cost of a missed churner.
  Threshold-dependent metrics (F1, precision) are intentionally omitted here;
  the operating point is fixed by the recall-driven threshold selection in §4.

### 2.2 Candidate Model Families

Three architecturally distinct families:

| Family | Model | Rationale |
|--------|-------|-----------|
| Linear | Logistic Regression | Baseline; interpretable; linear decision boundary |
| Bagging | Random Forest | Variance reduction; built-in feature importance |
| Boosting | XGBoost | Sequentially corrects errors; strong on tabular data |

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = ["roc_auc", "recall"]

candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE,
                                          verbosity=0, n_jobs=-1),
}

cv_results = {}
for name, clf in candidates.items():
    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote",        SMOTE(random_state=RANDOM_STATE)),
        ("classifier",   clf),
    ])
    scores = cross_validate(pipe, X_train, y_train,
                            cv=skf, scoring=SCORING, n_jobs=-1)
    cv_results[name] = {m: scores[f"test_{m}"] for m in SCORING}
    auc = scores["test_roc_auc"]
    print(f"{name:22}  ROC-AUC {auc.mean():.4f} ± {auc.std():.4f}")

print("\n5-fold CV complete.")

In [ ]:
# Results table
rows = []
for name, res in cv_results.items():
    row = {"Model": name}
    for m in SCORING:
        row[f"{m} mean"] = res[m].mean()
        row[f"{m} std"]  = res[m].std()
    rows.append(row)
cv_df = pd.DataFrame(rows).set_index("Model")

print("CV Results — mean ± std across 5 folds\n")
mean_cols = [c for c in cv_df.columns if "mean" in c]
for name, row in cv_df.iterrows():
    parts = "  |  ".join(
        f"{m.replace(' mean',''):>12} {row[m]:.4f}±{row[m.replace('mean','std')]:.4f}"
        for m in mean_cols
    )
    print(f"  {name:<22}  {parts}")

In [ ]:
models = list(cv_results.keys())
x = np.arange(len(models))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
b1 = ax.bar(x - w/2, [cv_results[m]["roc_auc"].mean() for m in models], w,
            yerr=[cv_results[m]["roc_auc"].std() for m in models],
            label="ROC-AUC", color="#2980b9", capsize=4, edgecolor="white")
b2 = ax.bar(x + w/2, [cv_results[m]["recall"].mean() for m in models], w,
            yerr=[cv_results[m]["recall"].std() for m in models],
            label="Recall", color="#e67e22", capsize=4, edgecolor="white")
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f"{h:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylim(0.50, 1.0); ax.set_ylabel("Score"); ax.legend()
ax.set_title("5-fold CV: ROC-AUC and Recall by Model Family", fontweight="bold")
plt.tight_layout(); plt.show()

### 2.3 Key Findings & Champion Declaration

**Interpretation:**
- **XGBoost** leads on ROC-AUC with low std across folds → best discriminative power, stable.
- **Random Forest** is second; bagging reduces variance but lacks boosting's error correction.
- **Logistic Regression** is the linear baseline; lower AUC confirms non-linear boundaries exist.

**Champion declared: XGBoost.**

*A single champion is declared now to prevent repeated hold-out access.
All further tuning (§3) targets this champion only, using CV on `train_dev`.*


In [ ]:
# Data-driven champion: pick the model with the highest mean ROC-AUC
CHAMPION_NAME = max(cv_results, key=lambda m: cv_results[m]["roc_auc"].mean())
print(f"Champion: {CHAMPION_NAME}  "
      f"(ROC-AUC {cv_results[CHAMPION_NAME]['roc_auc'].mean():.4f} "
      f"± {cv_results[CHAMPION_NAME]['roc_auc'].std():.4f})")

# Build baseline champion pipeline for ablation
_champ_clf_map = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE,
                                          verbosity=0, n_jobs=-1),
}
champion_pipe = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote",        SMOTE(random_state=RANDOM_STATE)),
    ("classifier",   _champ_clf_map[CHAMPION_NAME]),
])
print("Baseline champion pipeline built.")

---
## 3. Controlled Ablations & Tuning — Part C
**CRISP-DM Phase:** Modelling (refinement)

**Purpose:** Test ≤ 4 targeted hypotheses on the champion (XGBoost) only.
Each experiment changes **exactly one factor**; all others are held constant at the baseline.

Total CV fits: 4 experiments × 2 variants × 5 folds = **40 fits** (within ≤ 50 constraint).

### 3.1 Ablation Plan

| # | Hypothesis | Controlled Change |
|---|------------|-------------------|
| 1 | SMOTE improves minority-class recall without hurting AUC | SMOTE on vs off |
| 2 | More trees reduce variance | `n_estimators`: 50 vs 200 |
| 3 | Shallower trees reduce overfitting | `max_depth`: 3 vs 7 |
| 4 | Lower learning rate improves generalisation | `learning_rate`: 0.05 vs 0.30 |


In [ ]:
def run_ablation(exp_name, pipe, variant_label):
    scores = cross_validate(pipe, X_train, y_train,
                            cv=skf, scoring=["roc_auc", "recall"], n_jobs=-1)
    auc_m    = scores["test_roc_auc"].mean()
    auc_s    = scores["test_roc_auc"].std()
    recall_m = scores["test_recall"].mean()
    print(f"  {variant_label:<36}  ROC-AUC {auc_m:.4f}±{auc_s:.4f}  Recall {recall_m:.4f}")
    return {"exp": exp_name, "variant": variant_label,
            "auc_mean": auc_m, "auc_std": auc_s, "recall_mean": recall_m}

ablation_log = []

def _xgb(**kw):
    return XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE,
                         verbosity=0, n_jobs=-1, **kw)

def _pipe(clf, smote=True):
    steps = [("preprocessor", preprocessor)]
    if smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append(("classifier", clf))
    return ImbPipeline(steps)

# Exp 1 — SMOTE on vs off
print("Exp 1 — SMOTE on vs off:")
ablation_log.append(run_ablation("Exp1: SMOTE", _pipe(_xgb(), smote=True),  "SMOTE=on  (baseline)"))
ablation_log.append(run_ablation("Exp1: SMOTE", _pipe(_xgb(), smote=False), "SMOTE=off"))

In [ ]:
# Exp 2 — n_estimators
print("Exp 2 — n_estimators:")
for n, lbl in [(50, "n_estimators=50"), (200, "n_estimators=200")]:
    ablation_log.append(run_ablation("Exp2: n_estimators", _pipe(_xgb(n_estimators=n)), lbl))

In [ ]:
# Exp 3 — max_depth
print("Exp 3 — max_depth:")
for d, lbl in [(3, "max_depth=3"), (7, "max_depth=7")]:
    ablation_log.append(run_ablation("Exp3: max_depth", _pipe(_xgb(max_depth=d)), lbl))

In [ ]:
# Exp 4 — learning_rate
print("Exp 4 — learning_rate:")
for lr, lbl in [(0.05, "learning_rate=0.05"), (0.30, "learning_rate=0.30")]:
    ablation_log.append(run_ablation("Exp4: learning_rate", _pipe(_xgb(learning_rate=lr)), lbl))

In [ ]:
# Consolidated Ablation Log
abl_df = pd.DataFrame(ablation_log)

print("\n╔══════════════════════════════════════════════════════════════════════════════╗")
print("║                        CONSOLIDATED ABLATION LOG                            ║")
print("╚══════════════════════════════════════════════════════════════════════════════╝\n")

for exp in abl_df["exp"].unique():
    rows = abl_df[abl_df["exp"] == exp].reset_index(drop=True)
    best = rows["auc_mean"].idxmax()
    print(f"  {exp}")
    for i, row in rows.iterrows():
        marker = " ← winner" if i == best else ""
        print(f"    {row['variant']:<38}  "
              f"ROC-AUC {row['auc_mean']:.4f} ± {row['auc_std']:.4f}  "
              f"Recall {row['recall_mean']:.4f}{marker}")
    print()

In [ ]:
# Pick ablation winner per experiment and build refined champion config.
# Note: ablation experiments target XGBoost hyperparameters.
# If a non-XGB model was declared champion, these ablations still inform XGBoost
# and the final pipeline defaults to the CV-selected champion unchanged.
winners = abl_df.loc[abl_df.groupby("exp")["auc_mean"].idxmax()].set_index("exp")

use_smote = "SMOTE=off" not in winners.loc["Exp1: SMOTE", "variant"]
xgb_kwargs = {}
for exp, row in winners.iterrows():
    v = row["variant"]
    if "n_estimators=" in v:   xgb_kwargs["n_estimators"]  = int(v.split("=")[1])
    elif "max_depth=" in v:    xgb_kwargs["max_depth"]     = int(v.split("=")[1])
    elif "learning_rate=" in v: xgb_kwargs["learning_rate"] = float(v.split("=")[1])

if CHAMPION_NAME == "XGBoost":
    refined_clf = XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE,
                                 verbosity=0, n_jobs=-1, **xgb_kwargs)
    print(f"Refined XGBoost kwargs: {xgb_kwargs}")
else:
    # Non-XGB champion: keep baseline config; ablation findings are informational
    refined_clf = _champ_clf_map[CHAMPION_NAME]
    print(f"Champion is {CHAMPION_NAME} — XGB ablation results are informational.")
    print(f"Best XGB config for reference: {xgb_kwargs}")

steps = [("preprocessor", preprocessor)]
if use_smote:
    steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
steps.append(("classifier", refined_clf))
champion_pipe = ImbPipeline(steps)
print(f"SMOTE: {use_smote}")
print("Refined champion pipeline locked.")

In [ ]:
print("Running cross_val_predict for validation probabilities (train only)…")
val_proba = cross_val_predict(
    champion_pipe, X_train, y_train,
    cv=skf, method="predict_proba", n_jobs=-1
)[:, 1]
print(f"Done. {len(val_proba):,} validation predictions.")

---
## 4. Business Decision Making — Part E
**CRISP-DM Phase:** Deployment

### 4.1 Business Context

**Problem restatement:** Predict which customers will churn so the retention team can
intervene *before* the customer leaves.

**Asymmetric error costs:**

| Error | Consequence | Severity |
|-------|------------|----------|
| **False Negative** — miss a churner | Customer leaves uncontacted → lost lifetime value | **CRITICAL** |
| **False Positive** — flag a loyal customer | Wasted retention offer (~$20–50 voucher) | Moderate |

**Threshold direction:** FN > FP in cost → **lower** the classification threshold below 0.5
to flag more potential churners, accepting more false alarms to catch more real ones.

*Per brief guidelines, we argue the direction and a justifiable operating point —
not the exact mathematical optimum.*

In [ ]:
# Recall vs threshold on validation-fold probabilities (no hold-out access)
thresholds = np.arange(0.15, 0.75, 0.01)
records = []
for t in thresholds:
    preds = (val_proba >= t).astype(int)
    records.append({
        "threshold": round(t, 2),
        "recall":    recall_score(y_train, preds, zero_division=0),
    })
thresh_df = pd.DataFrame(records)

# Select tightest threshold that still achieves recall >= 0.80
valid = thresh_df[thresh_df["recall"] >= 0.80]
if len(valid):
    THRESHOLD = float(valid["threshold"].max())
else:
    THRESHOLD = float(thresh_df.loc[thresh_df["recall"].idxmax(), "threshold"])

THRESHOLD = round(THRESHOLD, 2)
sel_recall = thresh_df.loc[thresh_df["threshold"] == THRESHOLD, "recall"].values[0]
print(f"Selected threshold : {THRESHOLD}")
print(f"  Recall    : {sel_recall:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresh_df["threshold"], thresh_df["recall"], label="Recall", color="#e74c3c", lw=2)

ax.axvline(0.50,      color="gray",    linestyle="--", alpha=0.7, lw=1.5, label="Default 0.50")
ax.axvline(THRESHOLD, color="#e74c3c", linestyle="-",  lw=2,
           label=f"Selected {THRESHOLD} (recall ≥ 0.80)")

ax.set_xlabel("Classification Threshold", fontsize=11)
ax.set_ylabel("Recall", fontsize=11)
ax.set_title("Recall vs Threshold", fontweight="bold", fontsize=12)
ax.legend(fontsize=9); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

print(f"""
Justification (threshold = {THRESHOLD}):
  - Default 0.5 misses too many churners (recall too low for the business need).
  - At threshold {THRESHOLD}: recall >= 80% -- retention team alerted to >=80% of real churners.
  - FN cost (~$500+ lost LTV) >> FP cost (~$30 retention voucher) -> lower threshold is correct.
  - Exact optimum left to stakeholder negotiation; {THRESHOLD} is a justifiable operating point.
""")

### 4.2 Final Hold-Out Evaluation

⚠️ **The hold-out set is accessed exactly once — here.**
Champion is fit on full `train`; locked threshold is applied; metrics reported.

In [ ]:
print("Fitting champion on full train…")
champion_pipe.fit(X_train, y_train)

# ── HOLD-OUT EVALUATED ONCE ──────────────────────────────────────────────────
test_proba = champion_pipe.predict_proba(X_test)[:, 1]
test_pred  = (test_proba >= THRESHOLD).astype(int)

hold_auc  = roc_auc_score(y_test, test_proba)
hold_rec  = recall_score(y_test, test_pred)
hold_bacc = balanced_accuracy_score(y_test, test_pred)

print(f"\n══════════════════════════════════════════════════════")
print(f"  FINAL HOLD-OUT DEPLOYMENT METRICS  (threshold={THRESHOLD})")
print(f"══════════════════════════════════════════════════════")
print(f"  ROC-AUC           : {hold_auc:.4f}")
print(f"  Recall            : {hold_rec:.4f}")
print(f"  Balanced Accuracy : {hold_bacc:.4f}")
print(f"══════════════════════════════════════════════════════")

fig, ax = plt.subplots(figsize=(4, 4))
cm = confusion_matrix(y_test, test_pred)
ConfusionMatrixDisplay(cm, display_labels=["No Churn", "Churn"]).plot(
    ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Hold-Out Confusion Matrix\n(threshold={THRESHOLD})", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║                    EXECUTIVE BUSINESS SUMMARY                        ║
╚══════════════════════════════════════════════════════════════════════╝

Problem
  Telco churn erodes lifetime customer value. With ~22.5% of customers churning,
  proactive retention intervention is critical and cost-effective.

Solution
  XGBoost classifier (boosting family) in a leakage-safe ImbPipeline with SMOTE
  oversampling. Champion selected by 5-fold CV ROC-AUC comparison against
  Logistic Regression (linear) and Random Forest (bagging).

Hold-Out Deployment Performance  [threshold = {THRESHOLD}]
  · ROC-AUC {hold_auc:.3f} — the model correctly ranks churners above non-churners
    {hold_auc*100:.1f}% of the time across all possible thresholds.
  · Recall  {hold_rec:.3f} — catches {hold_rec*100:.1f}% of actual churners before they leave.

Business Decision
  Threshold lowered from 0.50 → {THRESHOLD} (direction: aggressive/lower).
  Rationale: FN cost (~$500+ lost LTV) >> FP cost (~$30 retention voucher).
  At this operating point the retention team captures ~{hold_rec*100:.0f}% of churners.
  Recommendation: tier retention offers by predicted probability (high/medium/low),
  concentrating premium spend on the highest-probability churners.
""")